# Lesson 2.4 — Fact-Checking and Validation Techniques

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Apply practical techniques to verify Claude's factual claims
- Use authoritative sources to confirm citations and numbers
- Record what you verified so others can trust the result

**You will produce:** A completed fact-check log (table) saved as a .md file

## Concepts

Validation turns a plausible draft into something you can rely on. Core techniques:

- **Check against the source of truth.** For a regulation, read the official text. For internal data, open the real file. (Sample Q1: verify the cited subsection against the official regulation.)
- **Isolate claims.** Pull out each factual claim, number, name, date, and citation; verify each one.
- **Provide the source to Claude.** Giving Claude the actual document to summarize reduces made-up details - but you still verify.
- **Triangulate.** For important facts, confirm with a second independent source.
- **Log it.** Record each claim, the source you checked, and the verdict.

This maps to Domain 2: *Apply fact-checking and validation techniques*.

> **Reference:** [Anthropic Docs - Reduce hallucinations](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations)

In [1]:
# !pip install anthropic   # run once if the SDK is not installed
# %pip install python-dotenv  # only if it isn't installed
import os
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Could not find .env from the notebook's working directory")

load_dotenv(env_path)
print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


def ask_claude(prompt, system=None, model="claude-sonnet-5-5", max_tokens=600):
    """Send a prompt to Claude. Falls back to a printed mock if no API key is set,
    so this notebook always runs. Set ANTHROPIC_API_KEY in your environment to use the real API.
    NOTE: Verify current model names at https://docs.anthropic.com (they change over time)."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("[No ANTHROPIC_API_KEY found - showing a MOCK response so the lesson still runs.]\n")
        return "[MOCK CLAUDE OUTPUT]\nThis is a placeholder. With a real key, Claude's answer appears here."
    try:
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
        kwargs = {"model": model, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        # Newer models may return thinking blocks first; keep only the text blocks
        return "".join(b.text for b in resp.content if b.type == "text")
    except Exception as e:
        print(f"[Could not reach the API: {e}. Showing a mock response.]\n")
        return "[MOCK CLAUDE OUTPUT] Placeholder response."

print("Helper ready. Call ask_claude('your prompt here').")

API key loaded: True
Helper ready. Call ask_claude('your prompt here').


## Code-Along: Step 1 — Extract the claims to check

Ask Claude to list its own checkable claims - a starting point, not proof.

In [2]:
summary = ask_claude('In 3 bullets, summarize why sleep matters for productivity, including any figures.')
claims = ask_claude(f'List each factual claim or number in the text below as a short checklist line.\n\n{summary}')
print(claims)

**Fact-check checklist**

- [ ] Even modest sleep loss impairs performance.
- [ ] Being awake 17-19 hours produces impairment comparable to a blood alcohol level of about 0.05% (Williamson & Feyer, 2000).
- [ ] Being awake 24 hours produces impairment closer to a blood alcohol level of 0.10% (Williamson & Feyer, 2000).
- [ ] Limiting sleep to 6 hours a night for two weeks produced deficits in attention and working memory (Van Dongen et al., 2003).
- [ ] Those deficits were similar to going up to two full nights without sleep (Van Dongen et al., 2003).
- [ ] People often don't notice how impaired they are.
- [ ] Adults need


## Code-Along: Step 2 — Build a fact-check log

A simple structure you fill in after checking each claim against a real source.

In [3]:
import io
rows = [
  ('Claim / number', 'Source checked', 'Verdict (OK / Wrong / Unverifiable)'),
  ('sleep is essential for health', 'https://www.sleepfoundation.org/how-sleep-works/why-do-we-need-sleep', 'Ok'),
]
md = '| ' + ' | '.join(rows[0]) + ' |\n| --- | --- | --- |\n'
for r in rows[1:]:
    md += '| ' + ' | '.join(r) + ' |\n'
print(md)

| Claim / number | Source checked | Verdict (OK / Wrong / Unverifiable) |
| --- | --- | --- |
| sleep is essential for health | https://www.sleepfoundation.org/how-sleep-works/why-do-we-need-sleep | Ok |



## Exercise

Complete and save a fact-check log for a real output. Replace the TODO rows with your actual claims and verdicts.

In [4]:
rows = [
  ('Saving is important for financial security', 'https://www.investopedia.com/articles/personal-finance/082415/importance-saving-money.asp', 'Ok'),
  ('Sex is good for health for men above 40 years old', 'https://www.healthline.com/health/healthy-sex/benefits-of-sex', 'Unverifiable'),
  ('Keeping pets is beneficial for emotional support', 'https://www.apa.org/news/press/releases/stress/2020/pets-coping-stress', 'Unverifiable'),
]
md = '# Fact-check log\n\n| Claim / number | Source checked | Verdict |\n| --- | --- | --- |\n'
for r in rows:
    md += '| ' + ' | '.join(r) + ' |\n'
with open('fact_check_log.md','w',encoding='utf-8') as f:
    f.write(md)
print('Saved fact_check_log.md')
print(md)

Saved fact_check_log.md
# Fact-check log

| Claim / number | Source checked | Verdict |
| --- | --- | --- |
| Saving is important for financial security | https://www.investopedia.com/articles/personal-finance/082415/importance-saving-money.asp | Ok |
| Sex is good for health for men above 40 years old | https://www.healthline.com/health/healthy-sex/benefits-of-sex | Unverifiable |
| Keeping pets is beneficial for emotional support | https://www.apa.org/news/press/releases/stress/2020/pets-coping-stress | Unverifiable |



## Key Takeaways

- Verify claims against the authoritative source, not against the model
- Break output into individual claims and check each one
- Giving Claude the real source reduces fabrication but doesn't remove the need to verify
- A fact-check log makes your diligence visible and repeatable

---

**Next:** [Lesson 2.5 — When Human Review Is Required](./lesson-2.5-when-human-review.ipynb)